In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )
include( "../recur.jl" )

base = "../data/results/";

In [ ]:
# Case and data folder, default parameters.
N = 100
μ = 1.0
ξ = 0.1;  ϕ = 0.25
ρ = 0.1

# Import default scale.
dparams = Params()
dscale = Scale( 1.0, ξ )
dnondim = Nondim( dparams; scale=dscale )

# Temporal variables.
T = 100_000
tlist = 0:5:T

# Range for density values.
μmin = -1;  μmax = 1
Nμ = 51;  Δμ = (μmax - μmin)/(Nμ-1)
μlist = round.( 10.0.^(μmin:Δμ:μmax), digits=6 )

# Adapatable time-step length.
smin = -3;  smax = 1
Ns = Nμ;  Δs = (smax - smin)/(Ns-1)
slist = round.( 10.0.^(smin:Δs:smax), digits=6 )

# Number of parameter combinations.
println( "Imported simulation data for $(Nμ) different values of μ." )
println( "Imported simulation data for $(Ns) different values of s0." )

In [ ]:
# Non-dimensional parameter list.
nondimlist = [Nondim(; s=s ) for s ∈ slist]
μc = criticalμ( nondimlist[1], N )

# Data folder name.
folderdata = [[findfolder( N, μ, nondim; base=base ) for nondim ∈ nondimlist] for μ ∈ μlist];

In [ ]:
# Import activity and determinism data.
ςdatalist = hcat( [[vec( readdlm( folder*"phase-order_T-$(round( defInt, T )).txt" ) ) for folder ∈ folderlist]
    for folderlist ∈ folderdata]... );

In [ ]:
# Determinism statistics.
μςdata = mean.( ςdatalist );

In [ ]:
cmap = cgrad( :ice, [1, 2]./3, rev=false )

# Generate heat map of determinism as a function of ρ and β.
plt = plot(; size=(300,275), dpi=100 )
plot!( plt; left_margin=0pt, bottom_margin=-5pt, right_margin=15pt )

# Plot the density-dependent phase transition.
heatmap!( plt, slist, μlist, hcat( μςdata... )'; cmap=cmap, colorbar=false )

plot!( plt, [slist[1],slist[end]], [μc, μc]; color=:gray45, lw=3, label="critical "*L"μ" )

plot!( plt; xlims=(slist[1],slist[end]), xscale=:log10 )
plot!( plt; ylims=(μlist[1],μlist[end]), yscale=:log10 )

plot!( plt; xlabel="speed constant, "*L"s_0", ylabel="density, "*L"μ", legend=:topright )

# function staticμ(N::defInt, params::Nondim)::defFloat
#     num = params.ρ^2*(1/(N*params.η) + params.τ)
#     den = 2π*params.β*params.s
#     return num/den
# end

# μ0list = [staticμ( N, nondim ) for nondim ∈ nondimlist]
# plot!( plt, slist, μ0list; color=:gray45, linestyle=:dot, lw=3, label="static "*L"μ" )

# saveplot( plt, figurefolder*"phase-coherence-spd-mu.png"; background=:white, dpi=600 )

In [ ]:
# Create dummy gradient data.
M = 1000
vals = reshape(range(0, 1, length=M), 1, :)

# Plot the gradient to make colorbar.
plt = plot(; size=(300,55), dpi=100 )

plot!( plt; left_margin=38.75pt, right_margin=15.25pt, top_margin=0pt, bottom_margin=15pt )

heatmap!( plt, vals; c=cmap, clims=(0,1),
    colorbar=false, ytick=false, yticks=false,
    framestyle=:box,
    xticks=( [1, round(Int,M/4), round(Int,N/2), round(Int,3M/4), M],
              ["0.0","0.25","0.50","0.75","1.0"] ) )

plot!( plt; xlabel="mean phase coherence, "*L"⟨ς_χ⟩" )

# saveplot( plt, figurefolder*"phase-coherence-spd-mu_colorbar.png"; background=:white, dpi=600 );